# Tunix Reasoning Trainer (Offline Mode)

This notebook trains a Gemma 2 model to "show its work" using Google's Tunix library on TPUs. 

**Offline Mode Enabled:** This notebook has been configured to run without internet access, using pre-downloaded dependencies and model weights.

## Step 1: Install Dependencies from Local Wheels

Since internet access is disabled, we install all required Python packages from the local Kaggle dataset.

**Paths:**
- We assume you uploaded your assets (packages and model) as a dataset named `tunix-offline-assets`.
- Kaggle mounts this at `/kaggle/input/tunix-offline-assets`.

**Tunix Installation Fix:**
- Direct installation from `/kaggle/input` fails because `pip` tries to write build artifacts to the read-only input folder.
- **Solution:** We copy the `tunix` source code to `/tmp/tunix` (a writable directory) and install from there.

In [ ]:
# Install dependencies from local wheels

# 1. Copy Tunix source to a writable directory to avoid 'Read-only file system' errors during build
!cp -r /kaggle/input/tunix-offline-assets/packages/tunix /tmp/tunix

# 2. Install Pip and Standard Libs
!pip install -U pip --no-index --find-links=/kaggle/input/tunix-offline-assets/packages
!pip install flax jax jaxlib --no-index --find-links=/kaggle/input/tunix-offline-assets/packages

# 3. Install Tunix from the WRITABLE /tmp directory
!pip install /tmp/tunix/.[tpu] --no-index --find-links=/kaggle/input/tunix-offline-assets/packages

## Step 2: Import Libraries

We import `jax` for high-performance computing on TPUs and `tunix` specific modules for SFT training with LoRA.

In [ ]:
import os
import json
import jax
import jax.numpy as jnp
import chex
import optax
import sentencepiece as spm
import qwix
from tunix.sft import peft_trainer
from tunix.sft.peft_trainer import PeftTrainer, TrainingConfig
from tunix.models.gemma import model as gemma_model
from tunix.models.gemma import params_safetensors

print(f"JAX devices: {jax.devices()}")

## Step 3: Configure Training (Offline Model)

We setup the data loader, model config, and trainer config.

**Crucial Change for Offline:**
- `MODEL_PATH`: Pointing to the specific local directory of the model weights.
- `DATASET_PATH`: Points to our local JSONL dataset.
- **Custom Data Loader**: Implemented `TextDataset` since `text_dataset.py` might vary in availability.

In [ ]:
# Model Configuration
# Ensure this path matches the actual mounted path of the model directory
MODEL_PATH = "/kaggle/input/tunix-offline-assets/model/gemma-2-2b-it" 
DATASET_PATH = "gsm8k_tunix_ready.jsonl"

# --- 1. Custom Data Loader ---
class TextDataset:
    def __init__(self, data_path, tokenizer_model_path, max_len=512):
        self.data = []
        with open(data_path, 'r') as f:
            for line in f:
                if line.strip():
                    self.data.append(json.loads(line))
        self.tokenizer = spm.SentencePieceProcessor()
        self.tokenizer.Load(tokenizer_model_path)
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __iter__(self):
        for item in self.data:
            # Construct simple instruction prompt
            text = f"User: {item.get('question', '')}\nModel: {item.get('answer', '')}"
            tokens = self.tokenizer.EncodeAsIds(text)
            if len(tokens) > self.max_len:
                tokens = tokens[:self.max_len]
            
            # Padding (0 is typically pad for SP depending on vocab, using 0 here simply)
            pad_len = self.max_len - len(tokens)
            input_ids = tokens + [0] * pad_len
            mask = [1] * len(tokens) + [0] * pad_len
            
            yield {
                "input_ids": jnp.array(input_ids, dtype=jnp.int32),
                "attention_mask": jnp.array(mask, dtype=jnp.int32)
            }

# --- 2. Initialize Data ---
tokenizer_path = os.path.join(MODEL_PATH, "tokenizer.model")
try:
    dataset = TextDataset(DATASET_PATH, tokenizer_path)
    print(f"Dataset loaded: {len(dataset)} examples")
except Exception as e:
    print(f"Warning: Dataset or Tokenizer not found ({e}). Skipping data load check.")
    dataset = []

# --- 3. Model Config ---
model_config = gemma_model.ModelConfig.gemma2_2b_it()

# --- 4. LoRA Provider ---
lora_provider = qwix.LoraProvider(
    module_path=".*w1|.*w2", 
    rank=8, 
    alpha=16
)

# --- 5. Training Config ---
training_config = TrainingConfig(
    eval_every_n_steps=10,
    max_steps=50,
    checkpoint_root_directory="/kaggle/working/checkpoints",
    gradient_accumulation_steps=1
)

# --- 6. Optimizer ---
optimizer = optax.adamw(learning_rate=1e-5)

print("Configuration ready.")

## Step 4: Run Training

Initialize the `PeftTrainer`, load the model weights, apply LoRA, and start training.

In [ ]:
# 1. Load Pretrained Weights
print("Loading model weights from safetensors...")
try:
    model_params = params_safetensors.create_model_from_safe_tensors(MODEL_PATH, model_config)
    print("Model weights loaded.")

    # 2. Apply LoRA
    print("Applying LoRA adapter...")
    # Dummy input required for shape inference in apply_lora_to_model
    dummy_input = {
        "x": jnp.ones((1, 1), dtype=jnp.int32),
        "positions": jnp.ones((1, 1), dtype=jnp.int32),
        "cache": None,
        "attention_mask": jnp.ones((1, 1, 1), dtype=jnp.bool),
    }
    lora_model = qwix.apply_lora_to_model(model_params, lora_provider, **dummy_input)
    
    # 3. Initialize Trainer
    print("Initializing PeftTrainer...")
    trainer = PeftTrainer(
        model=lora_model,
        optimizer=optimizer,
        training_config=training_config
    )
    
    # 4. Helper for Batches
    def batch_data_loader(ds, batch_size=4):
        batch = []
        for item in ds:
            batch.append(item)
            if len(batch) == batch_size:
                yield {k: jnp.stack([x[k] for x in batch]) for k in batch[0]}
                batch = []
    
    # 5. Train
    print("Starting Training Loop...")
    # Create separate iterators for train/eval
    train_loader = batch_data_loader(dataset)
    eval_loader = batch_data_loader(dataset) # Just reusing for demo
    
    trainer.train(train_loader, eval_loader)
    print("Training Complete.")
    
    # Save Model
    # trainer.save_checkpoint(...) # Check peft_trainer for save methods
except Exception as e:
    print(f"Training setup failed: {e}")

## Step 5: Inference (Optional)

Simple inference test.

In [ ]:
print("Inference would go here (requires generate implementation).")